In [1]:
!pip install langchain
!pip install openai
!pip install langchain_openai

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 974.0/974.0 kB 8.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 314.7/314.7 kB 11.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 125.2/125.2 kB 8.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.0/53.0 kB 3.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.7/142.7 kB 8.3 MB/s eta 0:00:00
  Attempting uninstall: packaging
    Found existing installation: packaging 24.1
    Uninstalling packaging-24.1:
      Successfully uninstalled packaging-24.1
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 325.5/325.5 kB 4.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 75.6/75.6 kB 6.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.9/77.9 kB 9.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.3/58.3 kB 7.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 22.7 MB/s eta 0:00:00


In [2]:
import os
import openai

from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [9]:
import os
import openai

from google.colab import userdata

os.environ['OPENAI_API_KEY'] = userdata.get('OPENAI_API_KEY')

In [10]:
# 모델의 응답 >> 문자열로 parsing 하는데 사용
# 언어모델(lm) 출력(응답) >> 텍스트 형식으로 변환

from langchain_core.output_parsers import StrOutputParser

# 대화형 프롬프트 템플릿을 정의
from langchain_core.prompts import ChatPromptTemplate

# open ai의 언어모델 사용, 대화형 응답을 생성
from langchain_openai import ChatOpenAI

# 객체 생성
chat_prompt_template = ChatPromptTemplate.from_template('Tell me a short joke about {topic}')
chat_model = ChatOpenAI()
output_parser = StrOutputParser()
# message 의 content 추출 >> string (문자)변환

# 체인 정의 (아직 호출하지 않음)
chain = chat_prompt_template | chat_model | output_parser

In [11]:
chain.invoke({'topic': 'ice cream'})
# invoke 역할
# >> 인자를 dict로 넘겨줌 (dict {k:v} >> {"사과":"과일"})
# 이유:  prompt_template 가 dict를 받기 때문

'Why did the ice cream cone go to therapy? Because it had too many sprinkles of anxiety!'

In [12]:
# 객체를 생성하지 않는 경우
# >> 체인 정의할 때 바로 생성자 이용

chain = chat_prompt_template | ChatOpenAI() | StrOutputParser()
chain.invoke({'topic': 'ice cream'})


'Why did the ice cream truck break down? \n\nBecause it had too many "scoops"!'

In [17]:
# dict 말고 'ice cream'만 넣고 싶을 때

from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate

from langchain_core.runnables import RunnablePassthrough, RunnableParallel, RunnableLambda

# runnables : 테스트 및 디버깅 용도
# RunnablePassthrough : 입력값을 그대로 출력값에 전달
# RunnableParallel : 여러 개의  Runnable 객체를 병렬로 실행
# RunnableLambda : 람다 함수나 임의의 함수 Runnable 감싸서 사용


chat_prompt_template = ChatPromptTemplate.from_template('Tell me a short joke about {topic}')
chat_model = ChatOpenAI()
output_parser = StrOutputParser()

# chain  = {'topic': RunnablePassthrough()} | chat_prompt_template | chat_model | output_parser

In [14]:
chain.invoke("ice cream")

# "ice cream" > RunnablePassthrough() > {'topic': RunnablePassthrough()} > prompt_template

'Why did the ice cream truck break down?\n\nBecause it had too many "scoops"!'

In [18]:
chain  = RunnablePassthrough() | chat_prompt_template | chat_model | output_parser

In [19]:
chain.invoke("ice cream")

'Why did the ice cream truck break down?\n\nBecause it had too many "scoops"!'

In [20]:
chain.invoke({'topic' :"ice cream"})

'Why did the ice cream truck break down?\n\nBecause it had too many "scoops"!'

In [21]:
class Runnable:

    def __init__(self, func):
        self.func = func

    # 이 메서드는 파이썬의 비트 OR 연산자(|)를 오버로드합니다.
    def __or__(self, other):

        ## 다른 함수가 이 함수의 결과를 활용
        def chained_func(*args, **kwargs):
            return other(self.func(*args, **kwargs))

        #러너블 객체로 감싸게 됨
        return Runnable(chained_func)

    def __call__(self, *args, **kwargs):
        return self.func(*args, **kwargs)

In [22]:
def add_five(x):
  return x + 5

def multipy_by_two(x):
  return x * 2

In [23]:
add_five(5)

10

In [24]:
multipy_by_two(5)

10

In [28]:
# Runnable 로 함수 감싸기
add_five = Runnable(add_five)
multiply_by_two = Runnable(multipy_by_two)

In [30]:
chain = add_five | multiply_by_two

chain(3)
# (3+5) * 2

16

In [31]:
chain = add_five.__or__(multiply_by_two)
chain(3)

16

In [32]:
def num2ko(x):
  return str(x) + '입니다.'

In [33]:
chain = add_five | multiply_by_two | num2ko

In [35]:
chain(3)

'16입니다.'

In [ ]:
# 템플릿 만드는 방법

# 미션 : 어떤 주제에 대해서 사용자 레벨에 맞게 선택된 언어로 대화 예시를 만드는 앱을 개발!
# 미션 : [어떤] 주제에 대해서 사용자 [레벨]에 맞게 [선택된 언어]로 대화 예시를 만드는 앱을 개발!

# 기획
# "Please generate dialogue sentences in English on the topic of health for a beginner level."
# "Please generate dialogue sentences in Korean on the topic of AI for an intermediate level."

# 양식과 변수를 분리하기
# "Please generate three sentences in a dialogue in (English) on the topic of (health) for a (beginner) level."
# "Please generate three sentences in a dialogue in (Korean) on the topic of (AI) for an (intermediate) level."

# "Please generate three sentences in a dialogue in {language} on the topic of {topic} for a/an {level} level."

# 양식 조정하기
# "Please generate three sentences in a dialogue in {language} on the topic of {topic} for a level of {level}."

In [36]:
from langchain_core.runnables import RunnablePassthrough

In [37]:
template =\
"Please generate three sentences in a dialogue in {language} on the topic of {topic} for a level of {level}"

In [ ]:
# cf) chatgpt
# [초보자 수준]으로 [저녁메뉴] 관련된 3개의 문장으로 구성된 대화지문 [한국어]로 생성해줘.

In [38]:
chat_prompt_template = ChatPromptTemplate.from_template(template)

In [39]:
chain = chat_prompt_template | chat_model | StrOutputParser()

In [41]:
# 변수는 dict 로 넣어야 함
output = chain.invoke( {"language" : "English", "topic": "dinner menu", "level":"beginner"} )
print(output)

Person 1: "What do you want for dinner tonight?"
Person 2: "I'm not sure, maybe we can have pasta?"
Person 1: "That sounds good, I'll make spaghetti with meatballs."


In [42]:
# 변수 중에 시스템에 입력해야 할 것과 사용자가 입력해야 할 것 분리
# language : 설정
# topic : 바뀐다
# level: 설정

def get_learning_language(_):
  print("###")
  print(_)
  print("in get_learning_language")
  print("###")
  return "English"


def get_learning_level(_):
  print("###")
  print(_)
  print("in get_learning_level")
  print("###")
  return "beginner"

# '_' : 함수 내에서 사용되지 않는 값을 받을 때
# 즉 여기서는 입력값을 사용하지 않고 항상 "beginner"를 반환

In [43]:
# 호출 방법 1

template =\
"Please generate three sentences in a dialogue in {language} on the topic of {topic} for a level of {level}"

prompt_template = ChatPromptTemplate.from_template(template)

chain = prompt_template | chat_model | StrOutputParser()

# invoke 하기 전에 필요한 정보를 dict 로 구성하는 방법
output =\
chain.invoke({'language': get_learning_language(''), 'topic': "travel", "level": get_learning_level('') })

print(output)

###

in get_learning_language
###
###

in get_learning_level
###
Person 1: Where do you want to go on your next vacation?
Person 2: I want to visit Paris, I've always dreamed of seeing the Eiffel Tower.
Person 1: That sounds amazing! I hope you have a great trip and take lots of pictures.


In [44]:
# 방법 2 Runnable 사용

from langchain_core.runnables import RunnablePassthrough

template =\
"Please generate three sentences in a dialogue in {language} on the topic of {topic} for a level of {level}"

chat_prompt_template = ChatPromptTemplate.from_template(template)

# 함수의 인자는 '무조건 1개' 이어야 함

chain = (
  RunnablePassthrough.assign(language=get_learning_language,
                           level=get_learning_level
                           )
  | chat_prompt_template
  | chat_model
  | StrOutputParser()
)

# {'topic': 'travel'} >> dict language, level 추가 >> {"topic",..."language", ...."level"...} dict 완성

output = chain.invoke({'topic': "travel"})   # 변수 1개

print(output)


###
{'topic': 'travel'}
in get_learning_language
###
###
{'topic': 'travel'}
in get_learning_level
###
Person 1: "Have you ever been on a plane before?"
Person 2: "No, I haven't. I'm excited to travel to new places."
Person 1: "Me too! I love exploring different countries and trying new foods."


In [45]:
chain

RunnableAssign(mapper={
  language: RunnableLambda(get_learning_language),
  level: RunnableLambda(get_learning_level)
})
| ChatPromptTemplate(input_variables=['language', 'level', 'topic'], messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['language', 'level', 'topic'], template='Please generate three sentences in a dialogue in {language} on the topic of {topic} for a level of {level}'))])
| ChatOpenAI(client=<openai.resources.chat.completions.Completions object at 0x7c51c118acb0>, async_client=<openai.resources.chat.completions.AsyncCompletions object at 0x7c51c1189420>, openai_api_key=SecretStr('**********'), openai_proxy='')
| StrOutputParser()

In [46]:
chain = (
  RunnablePassthrough.assign(language=get_learning_language,
                           level=get_learning_level
                           )
  | chat_prompt_template
  | chat_model
  | StrOutputParser()
)

# {'topic': 'travel'} >> dict language, level 추가 >> {"topic",..."language", ...."level"...} dict 완성

output = chain.invoke({'topic': "weather"})   # 변수 1개

print(output)

###
###
{'topic': 'weather'}
in get_learning_level{'topic': 'weather'}
in get_learning_language

###
###
Person 1: "Is it raining outside today?"
Person 2: "Yes, it's pouring down. I forgot my umbrella at home."
Person 1: "Oh no, that's too bad. I hope it stops soon."


In [47]:
chain

RunnableAssign(mapper={
  language: RunnableLambda(get_learning_language),
  level: RunnableLambda(get_learning_level)
})
| ChatPromptTemplate(input_variables=['language', 'level', 'topic'], messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['language', 'level', 'topic'], template='Please generate three sentences in a dialogue in {language} on the topic of {topic} for a level of {level}'))])
| ChatOpenAI(client=<openai.resources.chat.completions.Completions object at 0x7c51c118acb0>, async_client=<openai.resources.chat.completions.AsyncCompletions object at 0x7c51c1189420>, openai_api_key=SecretStr('**********'), openai_proxy='')
| StrOutputParser()